In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Load and prepare data — same as Day 4
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
df = df.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'])
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

X = df.drop(columns=['Survived'])
y = df['Survived']

# 5-Fold Cross Validation
model = RandomForestClassifier(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print("Accuracy per fold:", scores.round(4))
print(f"Mean accuracy:  {scores.mean():.4f}")
print(f"Std deviation:  {scores.std():.4f}")


Accuracy per fold: [0.8436 0.7865 0.7753 0.8202 0.8539]
Mean accuracy:  0.8159
Std deviation:  0.0308


In [2]:
from sklearn.model_selection import GridSearchCV

# Define the hyperparameters to try
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,          
    verbose=1           
)

grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_.round(4))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'max_depth': 8, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Best CV accuracy: 0.8428


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

best_model = grid_search.best_estimator_
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Final Test Accuracy:", round(best_model.score(X_test, y_test), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Final Test Accuracy: 0.7933

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179

Confusion Matrix:
[[97 13]
 [24 45]]


In [4]:
rf_balanced = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'    
)

scores_balanced = cross_val_score(rf_balanced, X, y, cv=cv, scoring='f1')
print("Balanced model F1:", scores_balanced.mean().round(4))

# Method 2 — optimize for F1 instead of accuracy in GridSearch
grid_search_f1 = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'min_samples_leaf': [1, 2, 4]
    },
    cv=cv,
    scoring='f1',        
    n_jobs=-1
)

grid_search_f1.fit(X, y)
print("Best F1 params:", grid_search_f1.best_params_)
print("Best F1 score:", grid_search_f1.best_score_.round(4))

Balanced model F1: 0.7696
Best F1 params: {'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 200}
Best F1 score: 0.7859


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    random_state=42,
    max_depth=6,
    min_samples_leaf=4,
    n_estimators=200
)

model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Gap:            {(train_acc - test_acc):.4f}")



Train accuracy: 0.8862
Test accuracy:  0.8045
Gap:            0.0818


In [15]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# Constrained model — same settings that fixed your overfitting
model = RandomForestClassifier(
    random_state=42,
    max_depth=6,
    min_samples_leaf=4,
    n_estimators=200
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print("CV scores per fold:", scores.round(4))
print(f"Mean accuracy:     {scores.mean():.4f}")
print(f"Std deviation:     {scores.std():.4f}")

CV scores per fold: [0.8547 0.809  0.8202 0.8258 0.8483]
Mean accuracy:     0.8316
Std deviation:     0.0173


In [17]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model 1 — optimized for accuracy
model_acc = RandomForestClassifier(
    random_state=42,
    max_depth=6,
    min_samples_leaf=4,
    n_estimators=200
)
model_acc.fit(X_train, y_train)
pred_acc = model_acc.predict(X_test)

# Model 2 — optimized for F1 (balanced classes)
model_f1 = RandomForestClassifier(
    random_state=42,
    max_depth=8,
    min_samples_leaf=1,
    n_estimators=200,
    class_weight='balanced'    # ← this is what makes it F1 optimized
)
model_f1.fit(X_train, y_train)
pred_f1 = model_f1.predict(X_test)

# Compare both
print("=== Accuracy-Optimized Model ===")
print(f"Accuracy: {accuracy_score(y_test, pred_acc):.4f}")
print(f"F1 Score: {f1_score(y_test, pred_acc):.4f}")

print("\n=== F1-Optimized Model ===")
print(f"Accuracy: {accuracy_score(y_test, pred_f1):.4f}")
print(f"F1 Score: {f1_score(y_test, pred_f1):.4f}")

=== Accuracy-Optimized Model ===
Accuracy: 0.8045
F1 Score: 0.7244

=== F1-Optimized Model ===
Accuracy: 0.7989
F1 Score: 0.7353
